In [ ]:
from google.colab import drive
drive.mount("/content/drive")

EXP  = "s03_gamma_var_subunits"
REPO = "/content/motionnet_cnn"
CKPT = f"/content/drive/MyDrive/NUIN/motionnet/ckpt/{EXP}"
LOG  = f"/content/drive/MyDrive/NUIN/motionnet/ckpt/{EXP}.log"

!if [ -d {REPO} ]; then cd {REPO} && git fetch -q origin && git checkout -q main && git pull -q origin main; \
 else git clone -q -b main https://github.com/guardomayas/motionnet.git {REPO}; fi
!pip install -q -e {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
!cd {REPO} && python scripts/sweep.py {EXP} --dry-run

In [ ]:
import subprocess

def sweep_running():
    return subprocess.run(["pgrep", "-f", f"scripts/sweep.py {EXP}"],
                          capture_output=True).returncode == 0

if sweep_running():
    print("already running")
else:
    subprocess.Popen(["python", "-u", "scripts/sweep.py", EXP], cwd=REPO,
                     stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)          # = setsid: survives kernel restarts
    print("launched")

In [ ]:
!ps aux | grep "[s]weep.py" || echo "NOT running"
!ls {CKPT}/*.pt 2>/dev/null | grep -v _last | wc -l
!ls {CKPT}/*.err 2>/dev/null || true
!tail -n 5 {LOG}


In [ ]:
# subprocess.run(["pkill", "-f", f"scripts/sweep.py {EXP}"])

In [ ]:
import pandas as pd
from pathlib import Path
# from motionnet.analysis import rf_summary, load_run, plot_layer1

D = Path(CKPT)
# rf_summary(D)
runs = pd.read_csv(D / "summary.csv")
# rfs  = pd.read_csv(D / "rf_summary.csv").merge(runs, on="tag")
runs.groupby(["loss.gamma_var", "model.num_subunits_1"])["nmse"].agg(["mean", "std"])

In [ ]:
"""Plots for a gamma_var x num_subunits_1 sweep (summary.csv from scripts/sweep.py).

    from motionnet.analysis.sweep_plots import plot_sweep
    fig = plot_sweep(D / "summary.csv")
    fig.savefig(D / "s02_summary.png", dpi=200, bbox_inches="tight")
"""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

G, N = "loss.gamma_var", "model.num_subunits_1"

# gamma is ordered (pooled -> local), so one hue light -> dark, not categorical hues
GAMMA_COLORS = ["#6da7ec", "#2a78d6", "#104281"]
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def _style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def _by_n1(ax, df, col, ylabel, gammas, colors, logy=False, label_ends=True):
    """Seeds as faint dots, mean +- sd as line; x = n1 on a log2 axis."""
    n1s = sorted(df[N].unique())
    for g, c in zip(gammas, colors):
        d = df[df[G] == g]
        m = d.groupby(N)[col].agg(["mean", "std"]).reindex(n1s)
        ax.scatter(d[N], d[col], s=14, color=c, alpha=0.35, lw=0, zorder=2)
        ax.errorbar(n1s, m["mean"], yerr=m["std"], color=c, lw=2, marker="o", ms=7,
                    mec="white", mew=1.5, capsize=0, elinewidth=1.2, zorder=3,
                    label=f"γ = {g:g}")
        if label_ends:
            ax.annotate(f"γ = {g:g}", (n1s[-1], m["mean"].iloc[-1]), xytext=(8, 0),
                        textcoords="offset points", va="center", fontsize=9, color=INK)
    ax.set_xscale("log", base=2)
    ax.set_xticks(n1s, [str(n) for n in n1s])
    ax.minorticks_off()
    ax.set_xlim(n1s[0] / 1.3, n1s[-1] * 1.9)            # room for end labels
    if logy:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("layer-1 channels  $n_1$", color=INK, fontsize=10)
    ax.set_ylabel(ylabel, color=INK, fontsize=10)
    _style(ax)


def plot_sweep(summary_csv, title=None):
    df = pd.read_csv(summary_csv)
    gammas = sorted(df[G].unique())
    colors = GAMMA_COLORS[:len(gammas)] if len(gammas) <= 3 else \
        plt.cm.Blues(np.linspace(0.45, 0.95, len(gammas)))
    n1s = sorted(df[N].unique())
    markers = dict(zip(n1s, ["o", "s", "D", "^", "v"]))

    fig, axs = plt.subplots(1, 3, figsize=(14, 4.3), layout="constrained")

    # (a) task performance
    _by_n1(axs[0], df, "nmse", "val nMSE  (1 = predict zero)", gammas, colors)
    axs[0].set_ylim(bottom=0)
    axs[0].set_title("Task error", loc="left", fontsize=11, color=INK)

    # (b) disagreement between local estimates
    _by_n1(axs[1], df, "svar", "spatial variance of local estimates", gammas, colors,
           logy=True)
    axs[1].set_title("Local estimates disagree", loc="left", fontsize=11, color=INK)

    # (c) cost vs error: one point per run
    ax = axs[2]
    for g, c in zip(gammas, colors):
        for n in n1s:
            d = df[(df[G] == g) & (df[N] == n)]
            ax.scatter(d["zE"], d["nmse"], s=46, color=c, marker=markers[n],
                       edgecolor="white", lw=1.2, zorder=3)
    ax.set_xlabel("layer-1 energy  zE", color=INK, fontsize=10)
    ax.set_ylabel("val nMSE", color=INK, fontsize=10)
    ax.set_ylim(bottom=0)
    ax.set_title("Error vs. layer-1 cost (one point per run)", loc="left",
                 fontsize=11, color=INK)
    _style(ax)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    # legend: color = gamma, shape = n1
    h = [plt.Line2D([], [], color=c, marker="o", ls="", ms=7, label=f"γ = {g:g}")
         for g, c in zip(gammas, colors)]
    h += [plt.Line2D([], [], color=MUTED, marker=markers[n], ls="", ms=7,
                     mfc="none", label=f"$n_1$ = {n}") for n in n1s]
    ax.legend(handles=h, frameon=False, fontsize=9, loc="upper left",
              bbox_to_anchor=(1.0, 1.0))

    fig.suptitle(title or "s02: locality (γ) × bottleneck ($n_1$)  —  dots = seeds, "
                 "line = mean ± s.d.", fontsize=11, color=MUTED, x=0.01, ha="left")
    return fig


def nmse_table(summary_csv):
    """mean ± sd nMSE as a gamma x n1 table of strings (for notes / slides)."""
    df = pd.read_csv(summary_csv)
    t = df.groupby([G, N])["nmse"].agg(["mean", "std"])
    s = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1)
    return s.unstack(N).rename_axis(index="γ", columns="n₁")

In [ ]:
# from motionnet.analysis.sweep_plots import plot_sweep, nmse_table
fig = plot_sweep(D / "summary.csv")
fig.savefig(D / "s02_summary.png", dpi=200, bbox_inches="tight")
nmse_table(D / "summary.csv")          # γ × n₁ table of "mean ± sd" strings